# 03 — BigQuery Extraction (Système Big Data)

**Task 1.7** — Sicurre Data Pipeline  
**Compétence C1** — Extraire des données à partir de sources hétérogènes (système big data)

---

## Objectif

Ce notebook satisfait l'exigence **« système big data »** du référentiel C1. Nous utilisons
Google BigQuery (free tier, 1 To/mois de requêtes) pour :

1. Charger le dataset HuggingFace **cybersectony/PhishingEmailDetectionv2.0** (~18 000 emails) dans BigQuery
2. Exécuter des requêtes SQL analytiques à grande échelle (window functions, FARM_FINGERPRINT, QUALIFY)
3. Extraire ~5 000 emails de phishing anglais dédupliqués
4. Exporter vers `data/raw/bigquery/` pour adaptation culturelle FR en aval (notebook 05)

**Auth :** `gcloud auth application-default login` (déjà configuré via CLI).  
**Deps :** gérées par `uv` via `pyproject.toml` (`google-cloud-bigquery`, `db-dtypes`, `pandas`, `datasets`).

In [ ]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import os
from collections.abc import Iterator
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from google.cloud import bigquery

# ── Load .env ────────────────────────────────────────────────────────
# Resolve relative to this notebook's location, not the kernel CWD
_NOTEBOOK_DIR: Path = Path(__file__).resolve().parent if "__file__" in dir() else Path("notebooks/bigdata")
ENV_PATH: Path = _NOTEBOOK_DIR.parent.parent / "data" / ".env"
if not ENV_PATH.exists():
    # Fallback: kernel CWD is project root
    ENV_PATH = Path("data/.env")
load_dotenv(ENV_PATH)

# ── Configuration (from data/.env) ───────────────────────────────────
PROJECT_ID: str = os.environ["SICURRE_GCP_PROJECT"]
DATASET_ID: str = os.environ["DATASET_ID"]
TABLE_NAME: str = "phishing_emails_en"
FULL_TABLE: str = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"
GCP_REGION: str = os.environ.get("SICURRE_GCP_REGION", "europe-west1")

OUTPUT_DIR: Path = Path("data/raw/bigdata/bigquery")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_TARGET: int = 5_000  # Number of phishing emails to extract
MIN_CONTENT_LENGTH: int = 50    # Filter out very short emails

print(f"Project     : {PROJECT_ID}")
print(f"Region      : {GCP_REGION}")
print(f"Dataset     : {DATASET_ID}")
print(f"Table       : {FULL_TABLE}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")
print(f"Target rows : {EXTRACTION_TARGET:,}")

Project     : sicurre
Region      : europe-west1
Dataset     : sicurre_dataset
Table       : sicurre.sicurre_dataset.phishing_emails_en
Output dir  : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/bigquery
Target rows : 5,000


## 1. Authenticate & Initialize BigQuery Client

Authentication uses **Application Default Credentials (ADC)** set via CLI:

```bash
gcloud auth application-default login
```

No service account key files needed for local development.

In [ ]:
# ── Initialize BigQuery Client ───────────────────────────────────────
client: bigquery.Client = bigquery.Client(project=PROJECT_ID)

# Verify connection
print("✅ Connected to BigQuery")
print(f"   Project : {client.project}")
print(f"   Location: {client.location or 'US (default)'}")

/Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/.venv/lib/python3.14/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Connected to BigQuery
   Project : sicurre
   Location: US (default)


## 2. Create Dataset & Load HuggingFace Data into BigQuery

We download **cybersectony/PhishingEmailDetectionv2.0** from HuggingFace Datasets,
convert to pandas, then upload to BigQuery using `load_table_from_dataframe()`.

This demonstrates the **BigQuery write path** — a key skill for the « système big data » competency.

In [3]:
# ── Create BigQuery Dataset (idempotent) ─────────────────────────────
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = GCP_REGION

dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"✅ Dataset ready: {dataset.full_dataset_id}")
print(f"   Location: {dataset.location}")

✅ Dataset ready: sicurre:sicurre_dataset
   Location: europe-west1


In [ ]:
# ── Load HuggingFace Dataset ─────────────────────────────────────────
hf_dataset = load_dataset(
    "cybersectony/PhishingEmailDetectionv2.0",
    split="train",
)

df_source_or_iter = hf_dataset.to_pandas()
if isinstance(df_source_or_iter, pd.DataFrame):
    df_source: pd.DataFrame = df_source_or_iter
elif isinstance(df_source_or_iter, Iterator):
    df_source = pd.concat(df_source_or_iter, ignore_index=True)
else:
    raise TypeError("Unexpected return type from hf_dataset.to_pandas()")

print(f"📊 HuggingFace dataset loaded: {df_source.shape[0]:,} rows × {df_source.shape[1]} cols")
print(f"   Columns: {list(df_source.columns)}")
print(f"\n── Label Distribution ──")
print(df_source["label"].value_counts())

📊 HuggingFace dataset loaded: 120,000 rows × 2 cols
   Columns: ['content', 'label']

── Label Distribution ──
label
3    53350
2    53157
0     6809
1     6684
Name: count, dtype: int64


In [ ]:
# ── Upload to BigQuery ───────────────────────────────────────────────
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_dataframe(
    dataframe=df_source,
    destination=FULL_TABLE,
    job_config=job_config,
)

# Wait for the job to complete
load_job.result()

# Verify row count in BigQuery
table = client.get_table(FULL_TABLE)
print(f"✅ Loaded {table.num_rows:,} rows into {FULL_TABLE}")

if table.num_bytes is None:
    print("   Size: unknown (num_bytes not available yet)")
else:
    print(f"   Size: {table.num_bytes / 1024 / 1024:.1f} MB")

/Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/.venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Loaded 120,000 rows into sicurre.sicurre_dataset.phishing_emails_en
   Size: 44.8 MB


## 3. Analyse du Dataset via SQL

Requêtes analytiques démontrant l'utilisation de BigQuery comme système big data :
- Agrégations (`GROUP BY`, `COUNT`)
- Distribution statistique (`AVG`, `STDDEV`, percentiles)
- Window functions (`ROW_NUMBER`, `QUALIFY`)
- Déduplication via `FARM_FINGERPRINT`

In [7]:
# ── 3a. Label Distribution (SQL) ─────────────────────────────────────
QUERY_LABELS: str = f"""
SELECT
    label,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM `{FULL_TABLE}`
GROUP BY label
ORDER BY count DESC
"""

df_labels = client.query(QUERY_LABELS).to_dataframe()
print("── Label Distribution (BigQuery) ──")
print(df_labels.to_string(index=False))

── Label Distribution (BigQuery) ──
 label  count   pct
     3  53350 44.46
     2  53157 44.30
     0   6809  5.67
     1   6684  5.57


/Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
# ── 3b. Content Length Statistics ────────────────────────────────────
QUERY_STATS: str = f"""
SELECT
    label,
    COUNT(*) AS n,
    ROUND(AVG(LENGTH(content)), 0)    AS avg_len,
    ROUND(STDDEV(LENGTH(content)), 0) AS std_len,
    MIN(LENGTH(content))              AS min_len,
    MAX(LENGTH(content))              AS max_len
FROM `{FULL_TABLE}`
GROUP BY label
"""

df_stats = client.query(QUERY_STATS).to_dataframe()
print("── Content Length Statistics ──")
print(df_stats.to_string(index=False))

── Content Length Statistics ──
 label     n  avg_len  std_len  min_len  max_len
     0  6809   4531.0 206500.0        5 17036692
     1  6684   1620.0   3300.0        1   134627
     2 53157     27.0      5.0       17       53
     3 53350     46.0     58.0       14     4274


/Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 4. Extract Phishing Emails (Deduplicated, Filtered)

SQL pipeline:
1. **Filter** label = 1 (phishing) AND content length > 50 chars
2. **Deduplicate** via `FARM_FINGERPRINT(content)` + `QUALIFY ROW_NUMBER()`
3. **Random sample** of 5 000 rows (`ORDER BY RAND()`)

This uses BigQuery-specific features (`QUALIFY`, `FARM_FINGERPRINT`) that demonstrate
mastery of a big data SQL dialect beyond standard SQL.

In [9]:
# ── Extract Deduplicated Phishing Emails ─────────────────────────────
QUERY_EXTRACT: str = f"""
SELECT
    content,
    label,
    LENGTH(content)                   AS content_length,
    FARM_FINGERPRINT(content)         AS content_hash
FROM `{FULL_TABLE}`
WHERE label = 1
  AND LENGTH(content) > {MIN_CONTENT_LENGTH}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY FARM_FINGERPRINT(content)
    ORDER BY LENGTH(content) DESC
) = 1
ORDER BY RAND()
LIMIT {EXTRACTION_TARGET}
"""

df_phishing: pd.DataFrame = client.query(QUERY_EXTRACT).to_dataframe()

print(f"✅ Extracted {df_phishing.shape[0]:,} deduplicated phishing emails")
print(f"   Avg length: {df_phishing['content_length'].mean():.0f} chars")
print(f"   Min length: {df_phishing['content_length'].min()} chars")
print(f"   Max length: {df_phishing['content_length'].max():,} chars")
print(f"\n── Preview (first 3 rows, truncated) ──")
df_phishing[["content", "content_length"]].head(3)

/Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Extracted 4,597 deduplicated phishing emails
   Avg length: 1752 chars
   Min length: 51 chars
   Max length: 134,627 chars

── Preview (first 3 rows, truncated) ──


,content,content_length
0,\nÃÂ \nÃÂ ÃÂ ÃÂ The \n Best M...,1742
1,_/ _/ _/ _/ _/ _/ _/ _/ _/ _/ _/ _/ _/ _/ _/ _...,1763
2,wanted ! home product assemblers ! looking for...,1092


## 5. Export to Local CSV

Save to `data/raw/bigquery/` with a timestamped filename for traceability.
This file feeds into the **cultural adaptation pipeline** (notebook 05).

Columns normalized to project convention: `text`, `label`, `source`, `language`.

In [10]:
# ── Export to CSV ────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
filename: str = f"bigquery_phishing_en_{df_phishing.shape[0]}_{timestamp}.csv"
output_path: Path = OUTPUT_DIR / filename

# Normalize columns to project convention
df_export: pd.DataFrame = df_phishing[["content", "label"]].copy()
df_export.columns = ["text", "label"]
df_export["source"] = "bigquery_cybersectony"
df_export["language"] = "en"

df_export.to_csv(output_path, index=False, encoding="utf-8")

file_size_mb: float = output_path.stat().st_size / 1024 / 1024
print(f"✅ Saved: {output_path}")
print(f"   Rows : {len(df_export):,}")
print(f"   Size : {file_size_mb:.2f} MB")
print(f"   Cols : {list(df_export.columns)}")

✅ Saved: ../data/raw/bigquery/bigquery_phishing_en_4597_20260228.csv
   Rows : 4,597
   Size : 7.93 MB
   Cols : ['text', 'label', 'source', 'language']


## 6. Query Cost Analysis

BigQuery free tier: **1 To/mois** de requêtes. Vérifions notre consommation.

In [11]:
# ── Query Cost Analysis ──────────────────────────────────────────────
QUERY_COST: str = f"""
SELECT
    job_id,
    creation_time,
    ROUND(total_bytes_processed / 1024 / 1024, 2) AS mb_processed,
    ROUND(total_bytes_billed / 1024 / 1024, 2)    AS mb_billed,
    cache_hit,
    state
FROM `region-{GCP_REGION}`.INFORMATION_SCHEMA.JOBS_BY_USER
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
  AND job_type = 'QUERY'
ORDER BY creation_time DESC
LIMIT 10
"""

try:
    df_cost = client.query(QUERY_COST).to_dataframe()
    total_mb_billed: float = df_cost["mb_billed"].sum()
    total_gb_billed: float = total_mb_billed / 1024.0
    # $5/TB after 1TB free tier
    estimated_cost_usd: float = max(0, total_gb_billed - 1024) * 5.0 / 1024

    print("── Recent Queries (last hour) ──")
    print(df_cost[["creation_time", "mb_processed", "mb_billed", "cache_hit"]].to_string(index=False))
    print(f"\n📊 Total billed this session: {total_mb_billed:.2f} MB ({total_gb_billed:.4f} GB)")
    print(f"💰 Estimated cost: ${estimated_cost_usd:.4f} USD (well within free tier)")
except Exception as e:
    print(f"⚠️  Could not retrieve cost info (might need INFORMATION_SCHEMA access): {e}")

── Recent Queries (last hour) ──
                   creation_time  mb_processed  mb_billed  cache_hit
2026-02-28 21:06:44.443000+00:00           NaN        NaN       <NA>
2026-02-28 21:06:37.923000+00:00         44.82       45.0      False

📊 Total billed this session: 45.00 MB (0.0439 GB)
💰 Estimated cost: $0.0000 USD (well within free tier)


/Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## Résumé

| Métrique | Valeur |
|----------|--------|
| Source | `cybersectony/PhishingEmailDetectionv2.0` via BigQuery |
| Rows extracted | ~5 000 phishing emails (EN) |
| Deduplication | `FARM_FINGERPRINT` + `QUALIFY ROW_NUMBER()` |
| Output | `data/raw/bigquery/bigquery_phishing_en_*.csv` |
| Next step | Notebook `05_cultural_adaptation.ipynb` (EN → FR adaptation) |

**Compétences démontrées :**
- **C1** : Extraction depuis un **système big data** (BigQuery)
- SQL avancé : window functions, QUALIFY, FARM_FINGERPRINT
- Pipeline reproductible : HuggingFace → BigQuery → CSV local